In [ ]:
## load all libraries

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_percentage_error,mean_absolute_percentage_error
from statsmodels.tsa.seasonal import STL
import numpy as np
from pandas import Series, DataFrame
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import statsmodels
from statsmodels.tsa.seasonal import seasonal_decompose
from pandas.plotting import register_matplotlib_converters
import pmdarima as pm
register_matplotlib_converters()
import warnings
import time
from numpy import array
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from numpy import array
import keras_tuner as kt
import tensorflow as tf
print(tf.__version__)
from tensorflow import keras
import keras_tuner as kt
from sklearn.preprocessing import MinMaxScaler
from keras.layers import Bidirectional
from keras.models import Sequential
from keras.preprocessing.sequence import TimeseriesGenerator
from keras.layers import Bidirectional
from tensorflow.keras import initializers
import random as rn
np.random.seed(123)
rn.seed(123)
tf.random.set_seed(123)
tf.keras.utils.set_random_seed(123)
keras.utils.set_random_seed(123)
warnings.filterwarnings('ignore')
import os


2.10.0


In [69]:
# helpers
## stl analysis
def stl_analysis(ts, filename,category):
    results = seasonal_decompose(ts,period=7)
    results.plot();
    plt.savefig(category+'_stl_analysis/'+filename+'.png')  
    plt.close()
    
## scale data and get its characterstics
def data_scaling_and_getting_characteristics(ts):
    train_all = ts.iloc[:int(len(ts)*0.9)]
    train = ts.iloc[:int(len(ts)*0.7)]
    val = ts.iloc[int(len(ts)*0.7):int(len(ts)*0.9)]
    test = ts.iloc[int(len(ts)*0.9):int(len(ts)*0.9)+21]
    MIN= np.min(train_all)
    MAX= np.max(train_all)
    test_mean=np.mean(test)['unit_sales']
    
    scaler = MinMaxScaler()
    scaler.fit(train_all)
    scaled_all = scaler.transform(ts)
    scaled_train = scaler.transform(train)
    scaled_train_all = scaler.transform(train_all)
    scaled_val = scaler.transform(val)
    scaled_test = scaler.transform(test)

    return scaled_train_all,scaled_train,scaled_val,scaled_val,scaled_test,MIN,MAX,test_mean

def arima_forecasting(ts,scaled_val,scaled_test):
    ## Train ARIMA m=12 cause periodicity is one month (12 period per year)
    arima = pm.auto_arima(ts,
                         max_p=7,max_d=2,max_q=7,max_Q=4,max_P=4,max_D=2,
                         n_fits =200,
                         trace=True,m=8,trend=[1,1,0,0],
                         random=True, maxiter =200,max_order = None,alpha=0.01,n_jobs=-1,
                         information_criterion='oob',out_of_sample_size=len(scaled_val),random_state =10)

    
    arima_prediction=arima.predict(n_periods=len(scaled_test))
    arima_fitted, conf_int = arima.predict_in_sample(return_conf_int=True, alpha=0.05)
       
    return arima_prediction, arima_fitted


def lstm_forecasting(scaled_train_all,scaled_train,scaled_val,scaled_test,category, filename):
    # shaping data
    n_features = 1
    n_input =9
    train_generator_all = TimeseriesGenerator(scaled_train_all, scaled_train_all, length=n_input, batch_size=2,shuffle=True)
    train_generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=2,shuffle=True)
    val_generator = TimeseriesGenerator(scaled_val, scaled_val, length=n_input, batch_size=2,shuffle=True)

    model = keras.Sequential()
    model.add(Bidirectional(LSTM(25, activation='relu', return_sequences=True), input_shape=(n_input, n_features)))
    model.add(Bidirectional(LSTM(25, activation='relu')))
    model.add(Dense(1))
    early_stopping_monitor = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=50,
        restore_best_weights=True
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),  loss='mse')
    
    history = model.fit_generator(train_generator, validation_data=val_generator,epochs=200,shuffle=True, verbose=1,callbacks=[early_stopping_monitor])
    plt.plot(history.history['loss'],label='loss')
    plt.plot(history.history['val_loss'],label='val_loss')
    plt.savefig(category+'_learning_curve_1/'+filename+'.png')  
    plt.legend()
    plt.close()
    
    val_loss_per_epoch = history.history['val_loss']
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print('Best epoch: %d' % (best_epoch))
    
    last_train_batch = scaled_train_all[-n_input:]
    last_train_batch = last_train_batch.reshape((1, n_input, n_features))
    model.predict(last_train_batch)

    # predicting training and test data
    lstm_predictions = []

    first_eval_batch = scaled_train_all[-n_input:]
    current_batch = first_eval_batch.reshape((1, n_input, n_features))

    for i in range(len(scaled_test)):
        current_pred = model.predict(current_batch,verbose=0)[0]
        lstm_predictions.append(current_pred) 
        #current_batch = np.append(current_batch[:,1:,:],[[scaled_test[i]]],axis=1)
        current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

    lstm_fit = []
    first_fit_batch = scaled_train_all[:n_input]
    current_batch = first_fit_batch.reshape((1, n_input, n_features))

    for i in range(n_input):
        lstm_fit.append(0)

    for i in range(len(scaled_train_all)-n_input):
        current_fit = model.predict(current_batch,verbose=0)[0]
        lstm_fit.append(current_fit) 
        #current_batch = np.append(current_batch[:,1:,:],[[scaled_train_all[i]]],axis=1)
        current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)
        
    return lstm_predictions,lstm_fit

def save_forecast_image(scaled_train_all,arima_fitted,arima_prediction,lstm_fit,lstm_predictions,category, filename,MIN,MAX):
    
    unscaled_train_all = scaled_train_all * int(MAX - MIN) + int(MIN)
    unscaled_test = scaled_test * int(MAX - MIN) + int(MIN)

    unscaled_arima_fitted = arima_fitted * int(MAX - MIN) + int(MIN)
    unscaled_arima_prediction = arima_prediction * int(MAX - MIN) + int(MIN)


    unscaled_lstm_fit=[x* int(MAX - MIN) for x in lstm_fit]
    unscaled_lstm_fit=[x+ int(MIN) for x in unscaled_lstm_fit]

    unscaled_lstm_predictions=[x* int(MAX - MIN) for x in lstm_predictions]
    unscaled_lstm_predictions=[x+ int(MIN) for x in unscaled_lstm_predictions]

    figure, ((ax1, ax2), (ax3, ax4) ) =plt.subplots(2, 2)
    figure.set_size_inches(20, 12)

    ax1.plot(unscaled_train_all,label = "Test")
    ax1.plot(unscaled_arima_fitted,label = "Prediction")
    ax1.legend()
    ax1.title.set_text('Train ARIMA')

    ax2.plot(unscaled_test,label = "Test")
    ax2.plot(unscaled_arima_prediction,label = "Prediction")
    ax2.legend()
    ax2.title.set_text('Test ARIMA')


    ax3.plot(unscaled_train_all,label = "Test")
    ax3.plot(unscaled_lstm_fit,label = "Prediction")
    ax3.legend()
    ax3.title.set_text('Train LSTM')

    ax4.plot(unscaled_test,label = "Test")
    ax4.plot(unscaled_lstm_predictions,label = "Prediction")
    ax4.legend()
    ax4.title.set_text('Test LSTM')
    
    plt.savefig(category+'_forecast/'+filename+'.png')  
    plt.close()

    return unscaled_test,unscaled_arima_prediction,unscaled_lstm_predictions

In [33]:
category ='intermittent'
os.mkdir(category+'_forecast')
os.mkdir(category+'_learning_curve_1')
os.mkdir(category+'_stl_analysis')


FileExistsError: [WinError 183] Impossible de créer un fichier déjà existant: 'intermittent_forecast'

In [ ]:
# assign directory
directory = 'dataset/intermittent_ts/'


stats = pd.DataFrame(columns=['arima_r2','arima_mse','arima_rmse','arima_mape','lstm_r2','lstm_mse','lstm_rmse','lstm_mape'])

scaler = MinMaxScaler()
# for each TS
for filename in os.listdir(directory):
    print(filename)
    f = os.path.join(directory, filename)
    # checking if it is a file
    if os.path.isfile(f):
        # read TS
        df3 = pd.read_csv(f)
        df3.columns = ['date', 'unit_sales']
        df3.set_index('date',inplace=True)
        
        # stl decomposition 
        stl_analysis(df3['unit_sales'],filename,category) 
        
        # save cv2, adi, sd, mean, stationarity and get scaled data
        scaled_train_all,scaled_train,scaled_val,scaled_val,scaled_test,MIN,MAX,test_mean = data_scaling_and_getting_characteristics(df3)

        
        # perform arima forecasting 
        arima_prediction, arima_fitted= arima_forecasting(scaled_train_all,scaled_val,scaled_test)
           
        # perform LSTM forecasting 
        lstm_predictions,lstm_fit=lstm_forecasting(scaled_train_all,scaled_train,scaled_val,scaled_test,category, filename)
          
            
        
        # save graphs to an image
        unscaled_test,unscaled_arima_prediction,unscaled_lstm_predictions=save_forecast_image(scaled_train_all,arima_fitted,arima_prediction,lstm_fit,lstm_predictions,category, filename,MIN,MAX)
               
            
        lstm_r2 = r2_score(scaled_test,lstm_predictions)
        lstm_mse= mean_squared_error(scaled_test,lstm_predictions,squared=True)
        lstm_rmse = mean_squared_error(scaled_test,lstm_predictions, squared=False)
        lstm_mape = mean_absolute_percentage_error(scaled_test,lstm_predictions)

        arima_r2 = r2_score(scaled_test,arima_prediction)
        arima_mse= mean_squared_error(scaled_test,arima_prediction,squared=True)
        arima_rmse = mean_squared_error(scaled_test,arima_prediction, squared=False)
        arima_mape = mean_absolute_percentage_error(scaled_test,arima_prediction)
        
        values_to_add = {'test_mean': test_mean,'arima_r2': arima_r2,'arima_mse': arima_mse,'arima_rmse': arima_rmse,'arima_mape': arima_mape,'lstm_r2': lstm_r2,'lstm_mse': lstm_mse,'lstm_rmse': lstm_rmse,'lstm_mape': lstm_mape }
        row_to_add = pd.Series(values_to_add, name='x')
        stats = stats.append(row_to_add)


27_222879.csv
Performing stepwise search to minimize oob
 ARIMA(2,0,2)(1,0,1)[8] intercept   : OOB=0.173, Time=1.87 sec
 ARIMA(0,0,0)(0,0,0)[8] intercept   : OOB=0.084, Time=0.07 sec
 ARIMA(1,0,0)(1,0,0)[8] intercept   : OOB=0.085, Time=0.99 sec
 ARIMA(0,0,1)(0,0,1)[8] intercept   : OOB=0.084, Time=0.79 sec
 ARIMA(0,0,0)(0,0,0)[8]             : OOB=0.084, Time=0.08 sec
 ARIMA(0,0,1)(0,0,0)[8] intercept   : OOB=0.083, Time=0.29 sec
 ARIMA(0,0,1)(1,0,0)[8] intercept   : OOB=0.084, Time=0.90 sec
 ARIMA(0,0,1)(1,0,1)[8] intercept   : OOB=0.165, Time=0.77 sec
 ARIMA(1,0,1)(0,0,0)[8] intercept   : OOB=0.092, Time=0.35 sec
 ARIMA(0,0,2)(0,0,0)[8] intercept   : OOB=0.083, Time=0.28 sec
 ARIMA(0,0,2)(1,0,0)[8] intercept   : OOB=0.124, Time=0.48 sec
 ARIMA(0,0,2)(0,0,1)[8] intercept   : OOB=0.084, Time=0.42 sec
 ARIMA(0,0,2)(1,0,1)[8] intercept   : OOB=0.164, Time=1.30 sec
 ARIMA(1,0,2)(0,0,0)[8] intercept   : OOB=0.090, Time=0.29 sec
 ARIMA(0,0,3)(0,0,0)[8] intercept   : OOB=0.087, Time=0.43 se

In [71]:
stats.to_csv(category+'.csv')  
stats

,arima_r2,arima_mse,arima_rmse,arima_mape,lstm_r2,lstm_mse,lstm_rmse,lstm_mape,test_mean
x,-1.048313,0.064958,0.254868,5.203803e-01,0.129706,0.027599,0.166131,6.121500e-01,10.904762
x,-1.483199,0.058661,0.242200,4.408611e-01,-0.091986,0.025796,0.160612,4.625222e-01,14.571429
x,-1.804671,0.055315,0.235192,1.089227e+14,-2.658337,0.072152,0.268611,7.253205e+14,5.238095
x,-0.205048,0.057006,0.238760,1.451937e+14,0.508408,0.023255,0.152497,1.786537e+13,11.380952
x,-0.197544,0.056040,0.236729,2.000534e+14,0.612632,0.018127,0.134638,9.691397e+12,22.904762
x,-0.850596,0.177413,0.421205,2.231439e+13,0.187883,0.077856,0.279027,4.579344e+14,16.190476
x,-0.159455,0.050054,0.223727,3.954265e+14,-0.245823,0.053782,0.231910,3.943207e+14,21.238095
x,-0.468336,0.064491,0.253950,4.251029e-01,-0.213808,0.053312,0.230893,9.699061e-01,26.095238
x,0.087500,0.040567,0.201413,3.134185e+14,0.319314,0.030261,0.173958,4.886609e+13,8.714286
x,-1.275876,0.061411,0.247812,4.732114e-01,0.172997,0.022315,0.149383,5.602989e-01,27.047619


In [72]:
stats[["arima_mse", "arima_rmse","lstm_mse", "lstm_rmse"]]

,arima_mse,arima_rmse,lstm_mse,lstm_rmse
x,0.064958,0.254868,0.027599,0.166131
x,0.058661,0.242200,0.025796,0.160612
x,0.055315,0.235192,0.072152,0.268611
x,0.057006,0.238760,0.023255,0.152497
x,0.056040,0.236729,0.018127,0.134638
x,0.177413,0.421205,0.077856,0.279027
x,0.050054,0.223727,0.053782,0.231910
x,0.064491,0.253950,0.053312,0.230893
x,0.040567,0.201413,0.030261,0.173958
x,0.061411,0.247812,0.022315,0.149383


In [73]:
print("arima mse : "+str(np.mean(stats['arima_mse'])))
print("lstm mse : "+str(np.mean(stats['lstm_mse'])))


arima mse : 0.06859164603508836
lstm mse : 0.040445682838188134


In [68]:
print("arima rmse : "+str(np.mean(stats['arima_rmse'])))
print("lstm rmse : "+str(np.mean(stats['lstm_rmse'])))

arima rmse : 0.25320005183625016
lstm rmse : 0.21407492233064188
